# polars-leetspeak

A Polars **expression plugin** that converts string columns to and from leetspeak.

The transformation is a compiled Rust function that the Polars engine loads at runtime via FFI.
It is not a Python UDF: there is no `map_elements`, no per-row Python call, and no GIL involvement.

```
df.with_columns(leetspeak("text"))
        |
        v  register_plugin_function(...)   <- Python wrapper
        v  polars stores an FFI pointer in the expression tree
        v  #[polars_expr(output_type=String)] fn leetspeak(inputs: &[Series])
        v  apply_into_string_amortized     <- one reused buffer, not one String per row
        v  Series back across the FFI boundary
   new DataFrame column
```

## 1. Setup

Import the plugin alongside Polars. Importing the package also registers the `.leet` expression namespace.

In [1]:
import polars as pl
from polars_leetspeak import leetspeak, unleetspeak, SUBSTITUTIONS

print("polars:", pl.__version__)
print("substitutions:", SUBSTITUTIONS)

polars: 1.44.2
substitutions: {'a': '4', 'b': '8', 'e': '3', 'g': '6', 'i': '1', 'o': '0', 's': '5', 't': '7', 'z': '2'}


## 2. The substitution table

| letter | a | b | e | g | i | o | s | t | z |
| ------ | - | - | - | - | - | - | - | - | - |
| digit  | 4 | 8 | 3 | 6 | 1 | 0 | 5 | 7 | 2 |

Note `l` is deliberately **not** mapped. Canonical leetspeak sends both `i` and `l` to `1`,
which would make decoding ambiguous. Dropping `l` keeps the mapping collision-free, so
`unleetspeak` is a genuine inverse. The trade-off: `hello` becomes `h3ll0`, not `h3110`.

## 3. Basic encoding

In [2]:
df = pl.DataFrame({"text": ["hello", "polars", "leetspeak", "base"]})

df.with_columns(leet=leetspeak("text"))

text,leet
str,str
"""hello""","""h3ll0"""
"""polars""","""p0l4r5"""
"""leetspeak""","""l3375p34k"""
"""base""","""8453"""


## 4. Decoding, and the round-trip

Because no two letters share a digit, decoding recovers the original exactly — for lowercase input with no digits of its own.

In [3]:
df.with_columns(leet=leetspeak("text")).with_columns(back=unleetspeak("leet"))

text,leet,back
str,str,str
"""hello""","""h3ll0""","""hello"""
"""polars""","""p0l4r5""","""polars"""
"""leetspeak""","""l3375p34k""","""leetspeak"""
"""base""","""8453""","""base"""


### Where the round-trip does *not* hold

Two documented sharp edges: encoding folds case, and decoding rewrites digits that were in the original text.

In [4]:
edge = pl.DataFrame({"text": ["HELLO", "Polars", "h4x0r in 2024"]})

edge.with_columns(
    leet=leetspeak("text"),
).with_columns(
    back=unleetspeak("leet"),
)

text,leet,back
str,str,str
"""HELLO""","""H3LL0""","""HeLLo"""
"""Polars""","""P0l4r5""","""Polars"""
"""h4x0r in 2024""","""h4x0r 1n 2024""","""haxor in zoza"""


Reading the result:

- `HELLO` -> `H3LL0` -> `HeLLo`: only the *substituted* characters come back lowercase, so decoding a
  mixed-case string leaves it in a case it never had. The round-trip is clean only for lowercase input.
- `Polars` -> `P0l4r5` -> `Polars`: this one survives, but only by luck — the capital `P` is not in the
  substitution table, so nothing about its case had to be recovered.
- `h4x0r in 2024` -> `h4x0r 1n 2024` -> `haxor in zoza`: the `4`, `0` and `2` that were already in the
  text get decoded as if they were leetspeak. Decoding is only safe on text you encoded.

## 5. The expression namespace

Importing the package registers a `.leet` namespace, so this reads more like native Polars.

In [5]:
df.with_columns(
    encoded=pl.col("text").leet.encode(),
).with_columns(
    decoded=pl.col("encoded").leet.decode(),
)

text,encoded,decoded
str,str,str
"""hello""","""h3ll0""","""hello"""
"""polars""","""p0l4r5""","""polars"""
"""leetspeak""","""l3375p34k""","""leetspeak"""
"""base""","""8453""","""base"""


## 6. It is a real expression

The plugin is declared `is_elementwise=True`, which is a promise to the optimizer. That makes it composable with the rest of the expression API, usable in lazy queries, and legal inside `group_by`.

In [6]:
lf = pl.LazyFrame({
    "user":  ["ana", "ana", "bob", "bob", "cleo"],
    "word":  ["gates", "boss", "tiger", "zebra", "oasis"],
})

(
    lf.with_columns(leet=leetspeak("word"))
      .group_by("user")
      .agg(
          pl.col("leet").sort().str.join(", ").alias("leetspoken"),
          pl.len().alias("n"),
      )
      .sort("user")
      .collect()
)

user,leetspoken,n
str,str,u32
"""ana""","""64735, 8055""",2
"""bob""","""238r4, 7163r""",2
"""cleo""","""04515""",1


Composing with other string expressions, and checking the optimizer keeps it in the plan:

In [7]:
q = (
    pl.LazyFrame({"word": ["gates", "boss", "tiger"]})
      .with_columns(
          shout=leetspeak(pl.col("word").str.to_uppercase() + "!"),
          n_digits=leetspeak("word").str.count_matches(r"\d"),
      )
)
print(q.explain())
q.collect()

 WITH_COLUMNS:
 [col("word").str.to_uppercase().str.concat_horizontal(["!"])./Users/konstantinostsoum/Desktop/polars-leetspeak/polars_leetspeak/polars_leetspeak.abi3.so:leetspeak().alias("shout"), col("word")./Users/konstantinostsoum/Desktop/polars-leetspeak/polars_leetspeak/polars_leetspeak.abi3.so:leetspeak().str.count_matches(["\d"]).alias("n_digits")] 
  DF ["word"]; PROJECT */1 COLUMNS


word,shout,n_digits
str,str,u32
"""gates""","""64735!""",5
"""boss""","""8055!""",4
"""tiger""","""7163R!""",4


## 7. Nulls and non-ASCII

Nulls propagate as nulls, and characters outside the table — including multi-byte UTF-8 — pass through untouched.

In [8]:
pl.DataFrame({"text": ["hello", None, "café", "日本語 gets", "", "!@#$"]}).with_columns(
    leet=leetspeak("text")
)

text,leet
str,str
"""hello""","""h3ll0"""
null,null
"""café""","""c4fé"""
"""日本語 gets""","""日本語 6375"""
"""""",""""""
"""!@#$""","""!@#$"""


## 8. Correctness checks

Asserting the behaviour rather than eyeballing it.

In [9]:
from polars.testing import assert_series_equal

# encoding
got = pl.DataFrame({"t": ["hello", "polars", "base"]}).with_columns(x=leetspeak("t"))["x"]
assert_series_equal(got, pl.Series("x", ["h3ll0", "p0l4r5", "8453"]))

# round-trip on lowercase, digit-free input
words = ["hello", "polars", "leetspeak", "abcdefg", "zigzag", "oasis"]
back = pl.DataFrame({"t": words}).with_columns(x=unleetspeak(leetspeak("t")))["x"]
assert_series_equal(back, pl.Series("x", words))

# nulls propagate
nulls = pl.DataFrame({"t": ["hello", None]}).with_columns(x=leetspeak("t"))["x"]
assert_series_equal(nulls, pl.Series("x", ["h3ll0", None]))

# the mapping really is collision-free
assert len(set(SUBSTITUTIONS.values())) == len(SUBSTITUTIONS)

# non-string input is rejected by the Rust side
try:
    pl.DataFrame({"n": [1, 2]}).with_columns(x=leetspeak("n"))
    raise AssertionError("expected a failure on non-string input")
except Exception as e:
    print("non-string input correctly rejected:", type(e).__name__)

print("all checks passed")

non-string input correctly rejected: ComputeError
all checks passed


## 9. Throughput

A million rows, to show the work happens in Rust rather than row-by-row in Python.

> **Build in release mode before reading anything into these numbers.** `maturin develop` produces an
> unoptimized debug build, which measured barely faster than the Python UDF below. Use
> `maturin develop --release`.

In [10]:
import time
import random, string

random.seed(0)
big = pl.DataFrame({
    "text": ["".join(random.choices(string.ascii_lowercase, k=12)) for _ in range(1_000_000)]
})

t0 = time.perf_counter()
out = big.with_columns(leet=leetspeak("text"))
elapsed = time.perf_counter() - t0

print(f"{big.height:,} rows in {elapsed:.3f}s  ({big.height/elapsed/1e6:.1f}M rows/s)")
out.head(3)

1,000,000 rows in 0.074s  (13.5M rows/s)


text,leet
str,str
"""vtkgnkuhmpxn""","""v7k6nkuhmpxn"""
"""htqgxzvxisxr""","""h7q6x2vx15xr"""
"""mclpxzmwguoa""","""mclpx2mw6u04"""


Against a pure-Python `map_elements` UDF doing the identical substitution. Note this is a
deliberately *hard* baseline: `str.translate` is implemented in C, so it is close to the best a
Python-level UDF can do. The plugin's win here is real but modest; the larger gains come from
avoiding the GIL when combined with other parallel work, not from the substitution itself.

In [11]:
# derived from the plugin's own table, so the two implementations cannot drift
table = str.maketrans({**SUBSTITUTIONS, **{k.upper(): v for k, v in SUBSTITUTIONS.items()}})

t0 = time.perf_counter()
_ = big.with_columns(
    leet=pl.col("text").map_elements(lambda s: s.translate(table), return_dtype=pl.String)
)
py_elapsed = time.perf_counter() - t0

print(f"python UDF : {py_elapsed:.3f}s")
print(f"rust plugin: {elapsed:.3f}s")
print(f"speedup    : {py_elapsed/elapsed:.1f}x")

python UDF : 0.476s
rust plugin: 0.074s
speedup    : 6.4x


## Summary

- The plugin is a compiled Rust `cdylib`, dynamically linked by Polars at runtime.
- It behaves like any native expression: lazy, composable, usable in `group_by`, null-aware.
- `leetspeak` / `unleetspeak` are exact inverses for lowercase, digit-free text; case folding and
  pre-existing digits are the two documented places that breaks.
- Because the mapping is collision-free, `l` is left alone — the deliberate cost of a usable decoder.